In [0]:
import pandas as pd

In [0]:
columns = [
    "order_id",
    "customer_id",
    "product_id",
    "order_date",
    "qty",
    "unit_price",
    "payment_method",
    "order_status"
]

In [0]:
dbutils.fs.mkdirs(
    "/Volumes/ecommerce/raw/raw_data/incoming_orders/")

True

In [0]:
data1 = [
    ["ORD90001", "CST1479", "PRD0315", "2026-09-04", 2, 67666.10, "UPI", "Shipped"],
    ["ORD90002", "CST0093", "PRD0289", "2026-09-04", 1, 484419.64, "Net Banking", "Delivered"],
    ["ORD90003", "CST0386", "PRD0509", "2026-09-04", 3, 449794.29, "Credit Card", "Processing"]
]

df1 = pd.DataFrame(data1, columns=columns)

df1.to_csv(
    "/Volumes/ecommerce/raw/raw_data/incoming_orders/orders_new_01.csv",
    index=False
)

In [0]:
data2 = [
    ["ORD90004", "CST2782", "PRD0237", "2026-09-04", 4, 45392.84, "Debit Card", "Shipped"],
    ["ORD90005", "CST3041", "PRD0156", "2026-09-04", 2, 259678.66, "UPI", "Delivered"],
    ["ORD90006", "CST1037", "PRD0262", "2026-09-04", 1, 71701.07, "Cash on Delivery", "Cancelled"]
]

df2 = pd.DataFrame(data2, columns=columns)

df2.to_csv(
    "/Volumes/ecommerce/raw/raw_data/incoming_orders/orders_new_02.csv",
    index=False
)

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/ecommerce/raw/raw_data/incoming_orders/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/ecommerce/raw/raw_data/incoming_orders/orders_new_01.csv,orders_new_01.csv,285,1788594299000
dbfs:/Volumes/ecommerce/raw/raw_data/incoming_orders/orders_new_02.csv,orders_new_02.csv,288,1788594309000


### Configuring paths

In [0]:
catalog = 'ecommerce'
schema = 'raw'

In [0]:
SOURCE_PATH = "/Volumes/ecommerce/raw/raw_data/incoming_orders/"
SCHEMA_PATH = "/Volumes/ecommerce/raw/raw_data/schema/autoloader_orders/"
CHECKPOINT_PATH = "/Volumes/ecommerce/raw/raw_data/checkpoints/autoloader_orders/"


### Reading file using Autoloader

In [0]:
from pyspark.sql import functions as F

In [0]:
orders_autoloader = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("header", "true")
        .load(SOURCE_PATH)
)

### Writing to bronze layer

In [0]:
autoloader_bronze_query = (
    orders_autoloader.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable("ecommerce.raw.brz_orders_autoloader")
)

In [0]:
display(
    spark.table("ecommerce.raw.brz_orders_autoloader")
)

order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status,_rescued_data
ORD90004,CST2782,PRD0237,2026-09-04,4,45392.84,Debit Card,Shipped,null
ORD90005,CST3041,PRD0156,2026-09-04,2,259678.66,UPI,Delivered,null
ORD90006,CST1037,PRD0262,2026-09-04,1,71701.07,Cash on Delivery,Cancelled,null
ORD90001,CST1479,PRD0315,2026-09-04,2,67666.1,UPI,Shipped,null
ORD90002,CST0093,PRD0289,2026-09-04,1,484419.64,Net Banking,Delivered,null
ORD90003,CST0386,PRD0509,2026-09-04,3,449794.29,Credit Card,Processing,null


In [0]:
print(
    spark.table("ecommerce.raw.brz_orders_autoloader").count()
)

6
